In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from classification_training_reporter import TrainingReporter
from classification_lr_model import build_model_from_grid_params
from classification_dataset_preprocessing import make_preprocessing_pipeline, make_label_pipeline, make_training_pipeline, make_delta_pipeline, make_standarize_pipeline
from classification_lr_model import CustomLogisticRegressionModel


# Odczyt datasetu

In [2]:
df = pd.read_csv(os.path.join("../data", "ortodoncja.csv"))

# Podział danych na zbiór treningowy i testowy

In [3]:
pipeline = make_label_pipeline()
df_processed = pipeline.fit_transform(df)
label_encoder = pipeline.named_steps["encode_labels"].encoder

X = df_processed.drop(columns=["growth direction"])
y = df_processed["growth direction"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tpipeline = make_training_pipeline()
X_train, y_train = tpipeline.fit_resample(X_train,y_train)
X_test = tpipeline[:1].fit_transform(X_test)

# Inicjalizacja reportera do treningów

In [4]:
model = CustomLogisticRegressionModel()
reporter = TrainingReporter(model, X_train, X_test, y_train, y_test)

# Pierwszy trening

In [5]:
reporter.train()

Start training...
Training finished!
Train Accuracy: 0.8502  |  Test Accuracy: 0.7889
Train F1:       0.8492  |  Test F1:       0.7898
Train AUROC:    0.9617  |  Test AUROC:    0.8843
---------------------------------------------------


# Cross walidacja

In [6]:
reporter.run_cross_validation(cv=10)

Start cross validation...
Fold 0:
  Train Accuracy: 0.8446  |  Val Accuracy: 0.8167
  Train F1:       0.8432  |  Val F1:       0.8145
---------------------------------------------------
Fold 1:
  Train Accuracy: 0.8521  |  Val Accuracy: 0.8333
  Train F1:       0.8512  |  Val F1:       0.8314
---------------------------------------------------
Fold 2:
  Train Accuracy: 0.8483  |  Val Accuracy: 0.8667
  Train F1:       0.8473  |  Val F1:       0.8653
---------------------------------------------------
Fold 3:
  Train Accuracy: 0.8521  |  Val Accuracy: 0.8167
  Train F1:       0.8512  |  Val F1:       0.8170
---------------------------------------------------
Fold 4:
  Train Accuracy: 0.8523  |  Val Accuracy: 0.8136
  Train F1:       0.8509  |  Val F1:       0.8101
---------------------------------------------------
Fold 5:
  Train Accuracy: 0.8467  |  Val Accuracy: 0.8814
  Train F1:       0.8458  |  Val F1:       0.8784
---------------------------------------------------
Fold 6:
  Trai

# Randomized grid search

In [7]:
random_grid = reporter.run_randomized_search_lr(cv=5)

Start randomized grid search for Logistic Regression...
Randomized search finished!
Best params: {'l1_ratio': np.float64(0.7777777777777777), 'C': 10.0}
Best F1 score: 0.823861932422807
---------------------------------------------------


# Kroswalidacja po dostosowaniu hiperparametrów za pomocą Randomized Grid Search

In [8]:
model_RGS = build_model_from_grid_params(random_grid.best_params_)
reporter_RGS = TrainingReporter(model_RGS, X_train, X_test, y_train, y_test)
reporter_RGS.run_cross_validation(cv=10)

Start cross validation...
Fold 0:
  Train Accuracy: 0.8633  |  Val Accuracy: 0.8500
  Train F1:       0.8632  |  Val F1:       0.8499
---------------------------------------------------
Fold 1:
  Train Accuracy: 0.8596  |  Val Accuracy: 0.8000
  Train F1:       0.8593  |  Val F1:       0.7957
---------------------------------------------------
Fold 2:
  Train Accuracy: 0.8633  |  Val Accuracy: 0.8667
  Train F1:       0.8633  |  Val F1:       0.8653
---------------------------------------------------
Fold 3:
  Train Accuracy: 0.8727  |  Val Accuracy: 0.8500
  Train F1:       0.8726  |  Val F1:       0.8485
---------------------------------------------------
Fold 4:
  Train Accuracy: 0.8654  |  Val Accuracy: 0.8305
  Train F1:       0.8654  |  Val F1:       0.8288
---------------------------------------------------
Fold 5:
  Train Accuracy: 0.8561  |  Val Accuracy: 0.8814
  Train F1:       0.8556  |  Val F1:       0.8784
---------------------------------------------------
Fold 6:
  Trai

# Zadowalające wyniki